[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S16_cierre_eda.ipynb)

# Sesión 16 · Cierre del análisis exploratorio

**Módulo 4: Matplotlib** · ⏱️ Duración estimada: 60 minutos en el notebook, más el trabajo del proyecto y la publicación

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Ir de una pregunta de negocio a una tabla, de la tabla a un gráfico y del gráfico a una conclusión con números.
2. Elegir el gráfico según la pregunta: evolución, ranking, composición o comparación de tendencias.
3. Comparar entidades de distinto tamaño con una métrica normalizada y dos series de distinta escala con índices, en un solo eje.
4. Exportar un conjunto de gráficos listos para un informe o una publicación.

## 📋 Qué debes saber antes
Módulo 3 (pandas) y sesiones 14 y 15 (Matplotlib).

## 🧭 Cómo trabajar este notebook
Hoy practicas **el mismo flujo que harás en tu proyecto**, con un dataset **ficticio** que tiene su misma forma: denuncias contra entidades financieras, con año, entidad, producto, motivo y región. Las entidades se llaman "Entidad A" a "Entidad F" a propósito: no representan a ninguna real.

Cada pregunta tiene tres pasos: **tabla → gráfico → conclusión**. La conclusión es un texto que guardas en una variable: una o dos oraciones, con al menos un número, que alguien que no vio el gráfico pueda entender.

Después del notebook vienen el 🧱 **avance del proyecto** (P2 completo) y la guía del 📣 **Post 1**.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos de práctica, aplica el estilo de las sesiones 14 y 15 y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Genera un dataset ficticio con la misma forma que el del proyecto, aplica el estilo y carga los verificadores.
import copy
import hashlib
import math
import os
import re
import statistics
from collections import defaultdict

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, MultipleLocator, PercentFormatter, StrMethodFormatter

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})

# ---------- Datos de práctica (ficticios): denuncias contra entidades financieras ----------
ANIOS = [2021, 2022, 2023, 2024, 2025]
_CLIENTES_2021 = {"Entidad A": 2_400_000, "Entidad B": 1_600_000, "Entidad C": 900_000,
                  "Entidad D": 500_000, "Entidad E": 250_000, "Entidad F": 120_000}
_TASA = {"Entidad A": 1.8, "Entidad B": 2.0, "Entidad C": 2.2, "Entidad D": 1.6, "Entidad E": 4.8, "Entidad F": 3.0}   # por 10 mil clientes
_CRECE = {"Entidad A": 1.04, "Entidad B": 1.03, "Entidad C": 1.05, "Entidad D": 1.20, "Entidad E": 1.06, "Entidad F": 1.02}
_PRODUCTOS = ["tarjeta de crédito", "préstamo personal", "cuenta de ahorros", "seguro", "crédito hipotecario"]
_MOTIVOS = ["cobro indebido", "demora en atención", "información incorrecta", "negativa de servicio", "operación no reconocida"]
_REGIONES = ["Arequipa", "Cusco", "La Libertad", "Lima", "Piura"]
_MEZCLA = {  # probabilidad de cada motivo según la región
    "Arequipa": [0.30, 0.25, 0.15, 0.10, 0.20], "Cusco": [0.20, 0.40, 0.15, 0.15, 0.10],
    "La Libertad": [0.35, 0.20, 0.15, 0.10, 0.20], "Lima": [0.30, 0.15, 0.10, 0.10, 0.35],
    "Piura": [0.25, 0.30, 0.20, 0.15, 0.10]}

clientes_entidad = pd.DataFrame(
    [[e, a, int(c * 1.03 ** (a - 2021))] for e, c in _CLIENTES_2021.items() for a in ANIOS],
    columns=["entidad", "anio", "clientes"])
_filas, _id = [], 1
for _e in _CLIENTES_2021:
    for _a in ANIOS:
        _clientes = _CLIENTES_2021[_e] * 1.03 ** (_a - 2021)
        for _ in range(int(rng.poisson(_TASA[_e] * _clientes / 10_000 * _CRECE[_e] ** (_a - 2021)))):
            _reg = str(rng.choice(_REGIONES, p=[0.12, 0.08, 0.1, 0.6, 0.1]))
            _filas.append([_id, _a, _e, str(rng.choice(_PRODUCTOS, p=[0.4, 0.25, 0.15, 0.1, 0.1])),
                           str(rng.choice(_MOTIVOS, p=_MEZCLA[_reg])), _reg])
            _id += 1
denuncias = pd.DataFrame(_filas, columns=["id", "anio", "entidad", "producto", "motivo", "region"])
credito = pd.Series(np.round(85_000 * 1.07 ** np.arange(5) * rng.uniform(0.98, 1.02, 5), 0), index=ANIOS, name="credito")

_D = copy.deepcopy({"denuncias": denuncias, "clientes_entidad": clientes_entidad, "credito": credito})
_FILAS = [tuple(f) for f in _D["denuncias"].itertuples(index=False)]

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")


def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def _conteo(clave):
    c = defaultdict(int)
    for f in _FILAS:
        c[clave(f)] += 1
    return c


def _entidades():
    return sorted({f[2] for f in _FILAS})


def _conclusion(r, nombre, debe_mencionar=None):
    t = r.var(nombre)
    if t is _FALTA:
        return
    if not isinstance(t, str) or len(t.strip()) < 60:
        r.mal(f"`{nombre}` debería ser un texto de al menos 60 caracteres: una conclusión completa, no una etiqueta.")
    elif not re.search(r"\d", t):
        r.mal(f"`{nombre}` debería incluir al menos un número que la respalde.")
    elif debe_mencionar and debe_mencionar.lower() not in t.lower():
        r.mal(f"`{nombre}` debería nombrar al protagonista del gráfico (lo que resaltaste en azul).")
    else:
        r.ok(f"`{nombre}` es una conclusión con respaldo.")


def _crecimiento():
    c = _conteo(lambda f: (f[1], f[2]))
    return {e: c[(2025, e)] / c[(2021, e)] for e in _entidades()}


def check_pregunta_1():
    r = _Revision("Pregunta 1 · Evolución")
    c = _conteo(lambda f: (f[1], f[2]))
    ents = _entidades()
    _df(r, "tabla_anual", ents, [[c[(a, e)] for e in ents] for a in ANIOS],
        "cantidad de denuncias con años en las filas y entidades en las columnas", indice=ANIOS)
    crec = _crecimiento()
    top = max(crec, key=crec.get)
    ax = _grafico(r, "ax1")
    if ax is not None:
        lineas = ax.get_lines()
        datos = [l for l in lineas if len(l.get_ydata()) == 5]
        azules = [l for l in datos if _hex(l.get_color()) == AZUL]
        grises = [l for l in datos if _hex(l.get_color()) == GRIS]
        if len(datos) != len(ents):
            r.mal(f"`ax1` debería tener {len(ents)} líneas, una por entidad.")
        elif len(azules) != 1 or len(grises) != len(ents) - 1:
            r.mal("En `ax1`, una sola línea debería ir en `AZUL` (la que más creció) y las demás en `GRIS`.")
        elif not _cerca_lista(azules[0].get_ydata(), [c[(a, top)] for a in ANIOS]):
            r.mal("La línea azul de `ax1` debería ser la de la entidad cuyas denuncias más crecieron entre 2021 y 2025.")
        elif top not in [t.get_text() for t in ax.texts]:
            r.mal("Etiqueta la línea azul con el nombre de su entidad (`ax.text`).")
        else:
            r.ok("`ax1` destaca a la entidad que más creció.")
    _conclusion(r, "conclusion_1", top)
    r.fin()


def check_pregunta_2():
    r = _Revision("Pregunta 2 · Bruto frente a normalizado")
    c = _conteo(lambda f: (f[1], f[2]))
    cli = {(f[0], f[1]): f[2] for f in _D["clientes_entidad"].itertuples(index=False)}
    ents = _entidades()
    bruto = {e: c[(2025, e)] for e in ents}
    norm = {e: round(c[(2025, e)] / cli[(e, 2025)] * 10_000, 1) for e in ents}
    _ser(r, "ranking_bruto", sorted(bruto.values(), reverse=True), "denuncias de 2025 por entidad, de mayor a menor",
         indice=sorted(bruto, key=lambda e: -bruto[e]))
    _ser(r, "por_10mil", sorted(norm.values(), reverse=True), "denuncias de 2025 por cada 10 mil clientes, con 1 decimal, de mayor a menor",
         indice=sorted(norm, key=lambda e: -norm[e]), tol=0.051)
    axs = r.var("axs2")
    if axs is not _FALTA:
        if not isinstance(axs, np.ndarray) or axs.shape != (2,):
            r.mal("`axs2` debería tener 2 ejes lado a lado (`plt.subplots(1, 2, ...)`).")
        else:
            for nombre, ax, valores in (("axs2[0]", axs[0], bruto), ("axs2[1]", axs[1], norm)):
                barras = _barras(ax)
                orden = sorted(valores.items(), key=lambda kv: kv[1])
                esperado = [AZUL if i == len(orden) - 1 else GRIS for i in range(len(orden))]
                if len(barras) != len(orden) or not _cerca_lista([b.get_width() for b in barras], [v for _, v in orden], 0.051):
                    r.mal(f"`{nombre}` debería tener barras horizontales ordenadas con la mayor arriba.")
                elif [_hex(b.get_facecolor()) for b in barras] != esperado:
                    r.mal(f"En `{nombre}`, solo la barra de arriba debería ir en `AZUL` y el resto en `GRIS`.")
                else:
                    r.ok(f"`{nombre}` es correcto.")
    top_norm = max(norm, key=norm.get)
    _conclusion(r, "conclusion_2", top_norm)
    r.fin()


def check_pregunta_3():
    r = _Revision("Pregunta 3 · Motivos por región")
    c = _conteo(lambda f: (f[5], f[4]))
    regiones = sorted({f[5] for f in _FILAS})
    motivos = sorted({f[4] for f in _FILAS})
    tabla = []
    for reg in regiones:
        tot = sum(c[(reg, m)] for m in motivos)
        tabla.append([round(c[(reg, m)] / tot, 3) for m in motivos])
    _df(r, "motivo_region", motivos, tabla, "la proporción de cada motivo dentro de cada región, con 3 decimales", indice=regiones, tol=0.00051)
    ax = _grafico(r, "ax3")
    if ax is not None:
        if not ax.images:
            r.mal("`ax3` debería ser un mapa de calor hecho con `ax.imshow`.")
        else:
            valores = np.asarray(ax.images[0].get_array()).ravel().tolist()
            if not _cerca_lista(valores, [x for fila in tabla for x in fila], 0.00051):
                r.mal("El mapa de calor debería mostrar `motivo_region`: regiones en las filas y motivos en las columnas.")
            elif [t.get_text() for t in ax.get_yticklabels()] != regiones or [t.get_text() for t in ax.get_xticklabels()] != motivos:
                r.mal("Pon los nombres de las regiones en el eje y y los de los motivos en el eje x (`set_yticks` y `set_xticks`).")
            elif ax.images[0].get_cmap().name != "Blues":
                r.mal("Usa una escala de un solo color (`cmap=\"Blues\"`): más oscuro, más proporción.")
            elif len(ax.figure.axes) < 2:
                r.mal("Agrega una barra de color (`fig3.colorbar(...)`) para que se pueda leer la escala.")
            else:
                r.ok("`ax3` es un mapa de calor legible.")
    _conclusion(r, "conclusion_3")
    r.fin()


def check_pregunta_4():
    r = _Revision("Pregunta 4 · Denuncias y crédito")
    c = _conteo(lambda f: f[1])
    cr = _D["credito"].tolist()
    ind_d = [round(c[a] / c[2021] * 100, 1) for a in ANIOS]
    ind_c = [round(x / cr[0] * 100, 1) for x in cr]
    _df(r, "indices", ["denuncias", "credito"], [[a, b] for a, b in zip(ind_d, ind_c)],
        "cada columna dividida entre su valor de 2021, por 100, con 1 decimal", indice=ANIOS, tol=0.051)
    ax = _grafico(r, "ax4")
    if ax is not None:
        datos = [l for l in ax.get_lines() if len(l.get_ydata()) == 5]
        leyenda = ax.get_legend()
        referencia = [l for l in ax.get_lines() if len(set(map(float, l.get_ydata()))) == 1 and float(l.get_ydata()[0]) == 100.0]
        if len(datos) != 2 or not any(_cerca_lista(l.get_ydata(), ind_d, 0.051) for l in datos) or not any(_cerca_lista(l.get_ydata(), ind_c, 0.051) for l in datos):
            r.mal("`ax4` debería tener 2 líneas en el mismo eje: el índice de denuncias y el de crédito.")
        elif leyenda is None or sorted(t.get_text() for t in leyenda.get_texts()) != ["crédito", "denuncias"]:
            r.mal("`ax4` necesita una leyenda que diga \"denuncias\" y \"crédito\".")
        elif not referencia:
            r.mal("Agrega una línea horizontal en 100 (el punto de partida) para leer cuánto creció cada serie.")
        elif any(a is not ax and a.get_shared_x_axes().joined(a, ax) for a in ax.figure.axes):
            r.mal("No uses un segundo eje y: con índices, las dos series comparten la misma escala.")
        else:
            r.ok("`ax4` compara las dos series con índices en un solo eje.")
    _conclusion(r, "conclusion_4")
    r.fin()


def _png(r, ruta, ancho_min=1000):
    if not os.path.exists(ruta):
        r.mal(f"No encuentro `{ruta}`.")
        return False
    ancho = mpimg.imread(ruta).shape[1]
    if ancho < ancho_min:
        r.mal(f"`{ruta}` mide {ancho} píxeles de ancho; guárdalo con `dpi=200`.")
        return False
    return True


def check_exportar():
    r = _Revision("Exportar")
    archivos = ["figuras/01_evolucion.png", "figuras/02_bruto_vs_normalizado.png", "figuras/03_motivos_region.png",
                "figuras/04_denuncias_vs_credito.png"]
    if all([_png(r, a) for a in archivos]):
        r.ok("Los cuatro gráficos están guardados en `figuras/` con buena resolución.")
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    axs = r.var("axs_res")
    if axs is not _FALTA:
        if not isinstance(axs, np.ndarray) or axs.shape != (2, 2):
            r.mal("`axs_res` debería ser una matriz de 2 × 2 ejes.")
        else:
            ents = len(_entidades())
            revisiones = [
                (len([l for l in axs[0, 0].get_lines() if len(l.get_ydata()) == 5]) == ents, "axs_res[0, 0] debería tener la evolución por entidad"),
                (len(_barras(axs[0, 1])) == ents, "axs_res[0, 1] debería tener las barras por cada 10 mil clientes"),
                (bool(axs[1, 0].images), "axs_res[1, 0] debería tener el mapa de calor"),
                (len([l for l in axs[1, 1].get_lines() if len(l.get_ydata()) == 5]) == 2, "axs_res[1, 1] debería tener los dos índices"),
            ]
            fallas = [m for ok, m in revisiones if not ok]
            r.ok("La lámina tiene los cuatro gráficos en su lugar.") if not fallas else r.mal("; ".join(fallas) + ".")
    fig = r.var("fig_res")
    if fig is not _FALTA and isinstance(fig, mpl.figure.Figure):
        t = getattr(fig, "_suptitle", None)
        r.ok("La lámina tiene un título general.") if t is not None and t.get_text().strip() else r.mal("Falta el título general de la lámina.")
    if _png(r, "figuras/00_resumen.png", 1800):
        r.ok("`figuras/00_resumen.png` está guardada con buena resolución.")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    ruta = "figuras/hallazgos.md"
    if not os.path.exists(ruta):
        r.mal(f"No encuentro `{ruta}`.")
    else:
        texto = open(ruta, encoding="utf-8").read()
        faltan = [n for n in ("conclusion_1", "conclusion_2", "conclusion_3", "conclusion_4")
                  if isinstance(globals().get(n), str) and globals()[n].strip() not in texto]
        vinetas = [l for l in texto.splitlines() if l.startswith("- ")]
        if faltan:
            r.mal(f"En `{ruta}` faltan estas conclusiones: {faltan}.")
        elif len(vinetas) < 4:
            r.mal("Cada conclusión debería ir en su propia línea, empezando con \"- \".")
        elif not texto.lstrip().startswith("#"):
            r.mal("El archivo debería empezar con un título de Markdown (una línea que empiece con \"#\").")
        else:
            r.ok(f"`{ruta}` tiene el título y las cuatro conclusiones en viñetas.")
    r.fin()


print("✅ Setup listo. Datos de práctica generados, estilo aplicado y verificadores cargados.")

### 📦 Tus datos de hoy
- `denuncias`: una fila por denuncia (id, año, entidad, producto, motivo y región), de 2021 a 2025.
- `clientes_entidad`: la cantidad de clientes de cada entidad cada año, para normalizar.
- `credito`: el saldo de crédito del sistema cada año, en millones (una Series con los años como índice).
- `ANIOS`: la lista de años.

In [ ]:
print(denuncias.shape)
print(denuncias.head(), "\n")
print(clientes_entidad.head(), "\n")
print(credito)

---
## Pregunta 1 · ¿Cómo evolucionan las denuncias de cada entidad?

### 📘 Qué gráfico usar
**Evolución en el tiempo → líneas.** Con seis entidades, seis líneas de colores forman un espagueti. Si el mensaje es sobre **una** entidad, destácala en `AZUL`, deja las demás en `GRIS` como contexto y escribe su nombre junto a la línea. El título cuenta la conclusión.

In [ ]:
ejemplo = pd.DataFrame({"x": [10, 12, 11, 13], "y": [5, 8, 12, 18], "z": [9, 9, 10, 10]}, index=[2022, 2023, 2024, 2025])
crece = (ejemplo.iloc[-1] / ejemplo.iloc[0]).idxmax()          # la columna que más creció
fig_ej, ax_ej = plt.subplots(figsize=(6, 3))
for col in ejemplo.columns:
    ax_ej.plot(ejemplo.index, ejemplo[col], color=AZUL if col == crece else GRIS, linewidth=2.5 if col == crece else 1)
ax_ej.text(2025.1, ejemplo[crece].iloc[-1], crece, va="center")
ax_ej.set_xticks(ejemplo.index)
ax_ej.set_title(f"La serie {crece} se triplicó en tres años")

### ✍️ Tu turno · Pregunta 1
1. `tabla_anual`: la cantidad de denuncias con los años en las filas y las entidades en las columnas.
2. `fig1, ax1`: una línea por entidad, con la entidad cuyas denuncias **más crecieron** entre 2021 y 2025 (cociente 2025 / 2021) en `AZUL` y grosor 2.5, las demás en `GRIS` y grosor 1, y el nombre de la destacada junto a su último punto. Muestra un año por marca en el eje x y escribe un título que cuente la conclusión.
3. `conclusion_1`: tu conclusión en una o dos oraciones, con al menos un número. Menciona a la entidad destacada.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pregunta_1()

<details><summary>💡 Pista 1</summary>

La tabla sale de `pd.crosstab` con el año y la entidad. El crecimiento es la última fila entre la primera.
</details>

<details><summary>💡 Pista 2</summary>

`crece = (tabla_anual.loc[2025] / tabla_anual.loc[2021]).idxmax()`. Luego recorre `tabla_anual.columns` como en el ejemplo.
</details>

---
## Pregunta 2 · ¿Qué entidades tienen más denuncias, en bruto y según su tamaño?

### 📘 Qué gráfico usar
**Ranking → barras horizontales ordenadas.** La cantidad bruta de denuncias premia a las entidades grandes: más clientes, más denuncias. La comparación justa es una **tasa**: denuncias por cada 10 mil clientes. Poner los dos rankings **lado a lado** (dos gráficos, no dos ejes en uno) muestra cómo cambia la historia.

In [ ]:
bruto_ej = pd.Series({"Grande": 500, "Mediana": 200, "Chica": 90})
clientes_ej = pd.Series({"Grande": 2_000_000, "Mediana": 600_000, "Chica": 150_000})
tasa_ej = (bruto_ej / clientes_ej * 10_000).round(1)
print(tasa_ej.sort_values(ascending=False))       # el ranking se invierte

### ✍️ Tu turno · Pregunta 2
Solo con el año 2025:
1. `ranking_bruto`: la cantidad de denuncias por entidad, de mayor a menor.
2. `por_10mil`: las denuncias por cada 10 mil clientes de cada entidad, con 1 decimal, de mayor a menor.
3. `fig2, axs2`: dos gráficos lado a lado (tamaño 12 × 4) con barras horizontales, la mayor arriba y en `AZUL`, el resto en `GRIS`: a la izquierda `ranking_bruto` y a la derecha `por_10mil`. Ponle a cada uno un título que diga qué mide.
4. `conclusion_2`: tu conclusión. Menciona a la entidad con más denuncias por cada 10 mil clientes.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pregunta_2()

<details><summary>💡 Pista 1</summary>

Filtra 2025 en las dos tablas. Para dividir, pon las dos Series con la entidad como índice: `value_counts()` y `set_index("entidad")["clientes"]`.
</details>

<details><summary>💡 Pista 2</summary>

Arma una función `barras_destacadas(ax, serie)` que ordene de menor a mayor, pinte la última en `AZUL` y el resto en `GRIS`, y úsala en los dos ejes.
</details>

---
## Pregunta 3 · ¿Qué motivos pesan más en cada región?

### 📘 Qué gráfico usar
**Composición en dos dimensiones → mapa de calor.** Una tabla de proporciones (cada fila suma 1) se ve bien como mapa de calor: `ax.imshow(tabla.values, cmap="Blues")` pinta cada celda con un azul más oscuro cuanto mayor es el valor. Una **escala de un solo color** se lee sin esfuerzo; las escalas de arcoíris engañan. La barra de color (`fig.colorbar(imagen, ax=ax)`) dice cuánto vale cada tono, y `set_xticks` / `set_yticks` ponen los nombres.

Usa proporciones y no conteos: Lima tiene muchas más denuncias y, con conteos, su fila taparía todo lo demás.

In [ ]:
tabla_ej = pd.DataFrame({"a": [0.7, 0.2], "b": [0.3, 0.8]}, index=["fila 1", "fila 2"])
fig_ej, ax_ej = plt.subplots(figsize=(4, 2.5))
imagen_ej = ax_ej.imshow(tabla_ej.values, cmap="Blues")
ax_ej.set_xticks(range(len(tabla_ej.columns)), tabla_ej.columns)
ax_ej.set_yticks(range(len(tabla_ej.index)), tabla_ej.index)
ax_ej.grid(False)
fig_ej.colorbar(imagen_ej, ax=ax_ej)

### ✍️ Tu turno · Pregunta 3
1. `motivo_region`: la proporción de cada motivo dentro de cada región (filas: regiones; columnas: motivos), con 3 decimales.
2. `fig3, ax3`: un mapa de calor de `motivo_region` con `cmap="Blues"`, los nombres en los dos ejes (gira los motivos 30° con `rotation=30, ha="right"` si se enciman), sin grilla, con barra de color y un título que cuente la conclusión.
3. `conclusion_3`: tu conclusión: ¿qué región se diferencia del resto y en qué motivo?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pregunta_3()

<details><summary>💡 Pista 1</summary>

`pd.crosstab(..., normalize="index")` da las proporciones por fila.
</details>

<details><summary>💡 Pista 2</summary>

Busca en el mapa la celda más oscura que no esté en la columna de un motivo que es alto en todas partes: esa es la diferencia regional.
</details>

---
## Pregunta 4 · ¿Las denuncias crecen al ritmo del crédito?

### 📘 Qué gráfico usar
Las denuncias se cuentan en miles y el crédito en decenas de miles de millones: dos escalas muy distintas. La tentación es un gráfico con **dos ejes y**, y es un error: la alineación de las dos escalas es arbitraria y puede fabricar una relación que no existe. La solución son los **índices**: divide cada serie entre su primer valor y multiplica por 100. Las dos empiezan en 100 y comparten un solo eje, y la pendiente muestra cuál creció más rápido. Una línea gris en 100 marca el punto de partida.

In [ ]:
ventas_ej = pd.Series([200, 220, 260], index=[2023, 2024, 2025])
visitas_ej = pd.Series([50_000, 51_000, 52_000], index=[2023, 2024, 2025])
indices_ej = pd.DataFrame({"ventas": ventas_ej / ventas_ej.iloc[0] * 100, "visitas": visitas_ej / visitas_ej.iloc[0] * 100}).round(1)
print(indices_ej)            # las ventas crecieron 30 %; las visitas, 4 %

### ✍️ Tu turno · Pregunta 4
1. `indices`: un DataFrame con los años como índice y dos columnas: `denuncias` (el total de denuncias de cada año) y `credito`, cada una como índice con base 2021 = 100 y 1 decimal.
2. `fig4, ax4`: las dos líneas en **un solo eje**, con leyenda (`denuncias` y `crédito`), una línea `GRIS` en 100, un año por marca en el eje x y un título que cuente la conclusión.
3. `conclusion_4`: tu conclusión, con el crecimiento de cada serie en porcentaje.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pregunta_4()

<details><summary>💡 Pista 1</summary>

El total de denuncias por año es `denuncias["anio"].value_counts().sort_index()`.
</details>

<details><summary>💡 Pista 2</summary>

`indices = pd.DataFrame({"denuncias": total / total.iloc[0] * 100, "credito": credito / credito.iloc[0] * 100}).round(1)`. Pon `label=` en cada `plot` para la leyenda.
</details>

---
## Exportar los gráficos

### 📘 Concepto
Guarda cada gráfico con un nombre que diga qué es y un número que marque su orden en la historia. `os.makedirs("figuras", exist_ok=True)` crea la carpeta si no existe.

In [ ]:
os.makedirs("figuras_ej", exist_ok=True)
fig_ej.savefig("figuras_ej/ejemplo.png", dpi=200, bbox_inches="tight")
print(os.listdir("figuras_ej"))

### ✍️ Tu turno · Exportar
Crea la carpeta `figuras` y guarda, con 200 dpi y sin márgenes sobrantes:
- `fig1` como `figuras/01_evolucion.png`
- `fig2` como `figuras/02_bruto_vs_normalizado.png`
- `fig3` como `figuras/03_motivos_region.png`
- `fig4` como `figuras/04_denuncias_vs_credito.png`

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_exportar()

<details><summary>💡 Pista 1</summary>

Una línea `savefig` por figura.
</details>

<details><summary>💡 Pista 2</summary>

Puedes recorrer pares: `for fig, nombre in [(fig1, "01_evolucion"), ...]: fig.savefig(f"figuras/{nombre}.png", dpi=200, bbox_inches="tight")`.
</details>

---
## 🏋️ Reto final: la lámina resumen
Crea `fig_res, axs_res` (2 × 2, tamaño 14 × 9) con las cuatro respuestas en una sola imagen:
- `axs_res[0, 0]`: la evolución por entidad (con el mismo énfasis).
- `axs_res[0, 1]`: las denuncias por cada 10 mil clientes (con el mismo énfasis).
- `axs_res[1, 0]`: el mapa de calor de motivos por región.
- `axs_res[1, 1]`: los dos índices.

Ponle a cada panel un título corto, un título general a la lámina, ajusta los espacios y guárdala como `figuras/00_resumen.png` con 200 dpi. Es la imagen de portada de tu post.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Convierte en funciones el código de cada gráfico, con el eje como parámetro (`def evolucion(ax): ...`), y llámalas una vez por panel.
</details>

<details><summary>💡 Pista 2</summary>

La barra de color del mapa de calor se agrega con `fig_res.colorbar(imagen, ax=axs_res[1, 0])`. Si los títulos se enciman, acórtalos.
</details>

---
## 🚀 Nivel pro (opcional): los hallazgos en Markdown
Escribe con Python el archivo `figuras/hallazgos.md`: una primera línea de título (`# Hallazgos`) y debajo tus cuatro conclusiones, cada una en su propia línea empezando con `- `. Es el borrador de la sección de hallazgos del README de tu proyecto. Pista: `"\n".join(...)` y `open(ruta, "w", encoding="utf-8")`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## 🧱 Avance del proyecto: P2 completo · 4 a 6 gráficos con conclusiones de negocio

**Qué hacer**
1. En `notebooks/03_eda_visualizacion.ipynb`, parte de la tabla limpia de P2 y responde las preguntas 1 a 4 del proyecto con el flujo de hoy: **pregunta → tabla → gráfico → conclusión**. Entre 4 y 6 gráficos en total.
2. Para la pregunta 2, usa los datos de tamaño de la SBS (clientes o créditos por entidad) para normalizar. Muestra el ranking bruto y el normalizado lado a lado.
3. Para la pregunta 4, cruza con la serie de crédito del BCRP y compara con índices de base 100 en un solo eje, nunca con dos ejes y.
4. Aplica lo de las sesiones 14 y 15: un color por serie o énfasis en gris y azul, barras ordenadas y desde 0, ejes con unidades legibles, títulos que cuentan la conclusión y la fuente de los datos en una nota al pie (`fig.text(...)`).
5. Guarda los gráficos en `reports/figures/` con 200 dpi y nombres numerados.
6. Actualiza el README con una sección **Hallazgos**: una viñeta por gráfico, con un número que la respalde, y una sección **Limitaciones**: qué no pueden decir estos datos (por ejemplo, más denuncias no siempre significa peor servicio; puede significar clientes más informados).

**Por qué lo haría un analista**
Un análisis que no se comunica no existe para el negocio. Pasar de tablas a pocos gráficos con conclusiones explícitas obliga a decidir qué importa, y la normalización y las limitaciones muestran criterio: es lo que distingue un análisis de un conjunto de gráficos.

**Cómo debe verse el resultado**
Un notebook que se ejecuta de principio a fin, entre 4 y 6 figuras en `reports/figures/` que se entienden solas (con título-conclusión, unidades y fuente) y un README en el que alguien sin conocimientos técnicos entiende en dos minutos qué encontraste.

---
## 📣 Post 1: publica tu análisis exploratorio

Tu primera publicación cuenta lo que encontraste, no cómo programaste. Con tus gráficos y tu README ya tienes todo el material.

**Estructura sugerida**
1. **Gancho** (1 o 2 líneas): el hallazgo más sorprendente, con un número.
2. **Contexto**: qué datos usaste (fuente pública y licencia) y qué pregunta querías responder.
3. **Dos o tres hallazgos**, cada uno en una línea y con su número. Incluye el de la normalización: es lo que muestra criterio.
4. **Una limitación** honesta.
5. **Enlace** al repositorio y una pregunta abierta para invitar a comentar.

**Imágenes**: la lámina `00_resumen.png` como portada y, si la plataforma lo permite, 2 o 3 gráficos individuales. Míralos en el celular antes de publicar: si no se leen los títulos, agranda la letra.

**Antes de publicar, revisa que:**
- [ ] cada cifra coincide con tu notebook;
- [ ] citas la fuente de los datos y su licencia;
- [ ] no aparece ningún dato personal;
- [ ] los gráficos se leen en una pantalla pequeña;
- [ ] agregaste un texto alternativo a cada imagen, describiendo lo que muestra;
- [ ] el repositorio es público y el README está al día.

Escríbelo con tu voz: cuenta qué te sorprendió y qué aprendiste.

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Recorrer el flujo pregunta → tabla → gráfico → conclusión.
- [ ] Elegir el gráfico según la pregunta: líneas para evolución, barras ordenadas para rankings, mapa de calor para composición en dos dimensiones.
- [ ] Explicar por qué el ranking bruto engaña y calcular una tasa por cada 10 mil clientes.
- [ ] Comparar dos series de distinta escala con índices de base 100, sin dos ejes y.
- [ ] Escribir conclusiones con números que se entienden sin ver el gráfico.
- [ ] Exportar un conjunto de figuras listo para un informe o una publicación.

**Próximo módulo (S17):** Machine Learning, empezando por qué es el aprendizaje supervisado.